In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from collections import Counter
import math

In [3]:
df = pd.read_parquet("20240608.parquet")

print(df.shape)
df.head()

(287806, 37)


,DST_IP,DST_IP_SUBNET,DST_IP_VERSION,DST_ASN,DST_COUNTRY,DST_PORT,PROTOCOL,TIME_FIRST,TIME_LAST,DURATION,...,QUIC_TLS_EXT_TYPE,QUIC_PACKETS,PPI,PPI_LEN,PPI_DURATION,PPI_ROUNDTRIPS,PHIST_SRC_SIZES,PHIST_DST_SIZES,PHIST_SRC_IPT,PHIST_DST_IPT
0,cf3d0d6011,9962760d94,4,15169,US,443,17,2024-06-07 23:00:00+02:00,2024-06-07 23:00:05.069670+02:00,5.069670,...,"[0, 10, 16, 13, 51, 45, 43, 57]","[129, 129, 129, 132, 132, 132, 132, 132, 132, ...","[[0, 1, 1, 13, 0, 0, 0, 2, 0, 0, 0, 0, 2, 0, 5...",30,0.032,7,"[0, 0, 6, 1, 0, 2, 0, 10]","[0, 4, 0, 1, 0, 0, 13, 11]","[17, 0, 0, 0, 0, 0, 0, 1]","[21, 2, 0, 1, 1, 1, 1, 1]"
1,e5d823e4a9,1ee05db139,4,32934,CZ,443,17,2024-06-07 23:00:00+02:00,2024-06-07 23:00:04.192914+02:00,4.192914,...,"[64250, 0, 10, 16, 5, 13, 18, 51, 45, 43, 57, ...","[129, 129, 132, 132, 128, 129, 132, 132, 128, ...","[[0, 1, 0, 0, 0, 6, 0, 4, 0, 0, 0, 0, 0, 0, 0,...",30,3.918,5,"[0, 2, 9, 1, 0, 4, 0, 3]","[0, 0, 5, 2, 1, 2, 0, 5]","[15, 0, 0, 0, 0, 2, 0, 1]","[8, 3, 0, 0, 1, 1, 0, 1]"
2,192a9a5d35,4d5e6fae0e,4,13335,ZZ,443,17,2024-06-07 23:00:00+02:00,2024-06-07 23:00:00.171934+02:00,0.171934,...,"[43, 45, 16, 13, 65037, 27, 42, 10, 51, 57, 17...","[129, 133, 130, 128, 128, 128, 128, 132, 128, ...","[[0, 2, 0, 0, 0, 0, 0, 5, 0, 0, 0, 1, 0, 1, 0,...",19,0.172,6,"[0, 0, 5, 2, 0, 0, 1, 2]","[0, 5, 2, 0, 0, 0, 1, 1]","[8, 0, 0, 0, 1, 0, 0, 0]","[7, 0, 0, 0, 1, 0, 0, 0]"
3,8bdb8548b1,1ee05db139,4,32934,CZ,443,17,2024-06-07 23:00:00+02:00,2024-06-07 23:00:00.080040+02:00,0.080040,...,"[43, 16, 0, 57, 45, 10, 13, 51, 41]","[129, 129, 132, 128, 133, 128, 132, 128, 128, ...","[[0, 1, 0, 0, 63, 0, 0, 0, 0, 1, 15, 0, 0], [1...",13,0.080,3,"[0, 0, 0, 1, 0, 0, 0, 3]","[0, 0, 4, 2, 1, 1, 0, 1]","[1, 1, 0, 1, 0, 0, 0, 0]","[7, 0, 1, 0, 0, 0, 0, 0]"
4,8bdb8548b1,1ee05db139,4,32934,CZ,443,17,2024-06-07 23:00:00+02:00,2024-06-07 23:00:00.283959+02:00,0.283959,...,"[43, 10, 51, 13, 0, 16, 45, 65445, 41]","[129, 129, 132, 128, 129, 132, 128, 128, 128, ...","[[0, 2, 0, 0, 26, 3, 0, 0, 0, 0, 0, 1, 1, 9, 1...",23,0.284,5,"[0, 0, 3, 1, 0, 0, 0, 6]","[0, 0, 4, 2, 1, 2, 1, 3]","[6, 1, 0, 2, 0, 0, 0, 0]","[6, 4, 0, 2, 0, 0, 0, 0]"


In [4]:
ppi_features = df[
    [
        "PPI",
        "PPI_LEN",
        "PPI_DURATION",
        "PPI_ROUNDTRIPS"
    ]
].copy()

ppi_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 287806 entries, 0 to 287805
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   PPI             287806 non-null  object 
 1   PPI_LEN         287806 non-null  int64  
 2   PPI_DURATION    287806 non-null  float64
 3   PPI_ROUNDTRIPS  287806 non-null  int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 8.8+ MB


In [5]:
# Calculate the average inter-packet time for each flow
ppi_features["mean_ipt"] = ppi_features["PPI"].apply(
    lambda x: np.mean(x[0])
)

In [6]:
# Calculate the variability of inter-packet times
ppi_features["std_ipt"] = ppi_features["PPI"].apply(
    lambda x: np.std(x[0])
)

In [7]:
# Calculate the average packet size
ppi_features["mean_packet_size"] = ppi_features["PPI"].apply(
    lambda x: np.mean(x[2])
)

In [8]:
# Calculate packet size variability
ppi_features["std_packet_size"] = ppi_features["PPI"].apply(
    lambda x: np.std(x[2])
)

In [9]:
def direction_change_ratio(directions):

    # If there is only one packet, no direction change is possible
    if len(directions) < 2:
        return 0

    changes = 0

    # Compare every packet direction with the next one
    for i in range(len(directions) - 1):

        if directions[i] != directions[i + 1]:
            changes += 1

    # Return the proportion of direction changes
    return changes / (len(directions) - 1)

In [10]:
ppi_features["direction_change_ratio"] = (
    ppi_features["PPI"].apply(
        lambda x: direction_change_ratio(x[1])
    )
)

In [11]:
def forward_packet_ratio(directions):

    if len(directions) == 0:
        return 0

    forward_packets = np.sum(directions == 1)

    return forward_packets / len(directions)

In [12]:
ppi_features["forward_packet_ratio"] = (
    ppi_features["PPI"].apply(
        lambda x: forward_packet_ratio(x[1])
    )
)

In [13]:
ppi_features["log_ppi_duration"] = np.log1p(ppi_features["PPI_DURATION"])
ppi_features["log_mean_ipt"] = np.log1p(ppi_features["mean_ipt"])
ppi_features["log_std_ipt"] = np.log1p(ppi_features["std_ipt"])

In [14]:
ppi_final = ppi_features[
    [
        # Original Features
        "PPI_LEN",
        "PPI_ROUNDTRIPS",

        # Log-transformed Features
        "log_ppi_duration",
        "log_mean_ipt",
        "log_std_ipt",

        # Engineered Features
        "mean_packet_size",
        "std_packet_size",
        "direction_change_ratio",
        "forward_packet_ratio"
    ]
].copy()

print("Final PPI Feature Bank Shape:", ppi_final.shape)

ppi_final.head()

Final PPI Feature Bank Shape: (287806, 9)


,PPI_LEN,PPI_ROUNDTRIPS,log_ppi_duration,log_mean_ipt,log_std_ipt,mean_packet_size,std_packet_size,direction_change_ratio,forward_packet_ratio
0,30,7,0.031499,0.725937,1.249640,966.433333,496.378262,0.482759,0.466667
1,30,5,1.592902,4.879767,6.445720,431.633333,497.956055,0.310345,0.566667
2,19,6,0.158712,2.307834,3.386832,306.157895,459.089518,0.666667,0.526316
3,13,3,0.076961,1.967650,2.883625,444.230769,517.949085,0.416667,0.307692
4,23,5,0.249980,2.591354,3.277578,559.217391,538.014624,0.409091,0.434783


In [15]:
cid_features = df[
    [
        "DST_ASN",
        "QUIC_OCCID",
        "QUIC_OSCID",
        "QUIC_SCID",
        "QUIC_RETRY_SCID",
        "QUIC_SNI"
    ]
].copy()

cid_features.head()

,DST_ASN,QUIC_OCCID,QUIC_OSCID,QUIC_SCID,QUIC_RETRY_SCID,QUIC_SNI
0,15169,9bd6126a26eba5712f774fc43540836a,ed012341a460dcfc62149d71af0156e1,ed012341a460dcfc,,ssl.gstatic.com
1,32934,,b030de89c7e45912,8921006707c6241d,,web.facebook.com
2,13335,,0330d9b3b666aebe,01c882ee3a66fe950cc89deea966cb462c263807,,discord.com
3,32934,,125343a043343ea6,ae2d000819c7484c,,i.instagram.com
4,32934,,937cff181dc5cb0c,8c2d0027035862b5,,i.instagram.com


In [16]:
cid_features["occid_length"] = cid_features["QUIC_OCCID"].str.len()

cid_features["oscid_length"] = cid_features["QUIC_OSCID"].str.len()

cid_features["scid_length"] = cid_features["QUIC_SCID"].str.len()

cid_features["retry_length"] = cid_features["QUIC_RETRY_SCID"].str.len()

cid_features["sni_length"] = cid_features["QUIC_SNI"].str.len()

cid_features["sni_labels"] = (
    cid_features["QUIC_SNI"]
    .str.split(".")
    .str.len()
)

In [17]:
cid_features["retry_present"] = (cid_features["retry_length"] > 0).astype(int)

In [18]:
def shannon_entropy(text):

    if len(text) == 0:
        return 0.0

    counts = Counter(text)

    entropy = 0.0

    length = len(text)

    for count in counts.values():

        p = count / length

        entropy -= p * math.log2(p)

    return entropy

In [19]:
def normalized_entropy(text):

    if len(text) == 0:
        return 0.0

    H = shannon_entropy(text)

    Hmax = math.log2(min(len(text), 16))

    return H / Hmax if Hmax > 0 else 0.0

In [20]:
cid_features["occid_entropy"] = (cid_features["QUIC_OCCID"].apply(normalized_entropy))

cid_features["oscid_entropy"] = (cid_features["QUIC_OSCID"].apply(normalized_entropy))

cid_features["scid_entropy"] = (cid_features["QUIC_SCID"].apply(normalized_entropy))

In [21]:
cid_features[
    [
        "occid_entropy",
        "oscid_entropy",
        "scid_entropy"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
occid_entropy,287806.0,0.136307,0.322104,0.000000,0.000000,0.000000,0.00000,1.000000
oscid_entropy,287806.0,0.809913,0.056570,0.470714,0.769455,0.812500,0.84375,0.988205
scid_entropy,287806.0,0.804154,0.064572,0.000000,0.769455,0.800705,0.84375,1.000000


In [22]:
MIN_FLOWS = max(50, int(0.001 * len(cid_features)))

In [23]:
asn_groups = cid_features.groupby("DST_ASN")

In [24]:
asn_profile = asn_groups.agg({

    "scid_length": ["count", "median"],

    "scid_entropy": "median",

    "oscid_length": "median",

    "oscid_entropy": "median",

    "retry_present": "mean",

    "sni_length": "median",

    "sni_labels": "median"

})

In [25]:
asn_profile.columns = [

    "flows",
    "median_scid_length",
    "median_scid_entropy",
    "median_oscid_length",
    "median_oscid_entropy",
    "retry_rate",
    "median_sni_length",
    "median_sni_labels",

]

In [26]:
asn_profile = asn_profile[asn_profile["flows"] >= MIN_FLOWS]

In [27]:
asn_profile.head()

,flows,median_scid_length,median_scid_entropy,median_oscid_length,median_oscid_entropy,retry_rate,median_sni_length,median_sni_labels
DST_ASN,,,,,,,,
6185,607,40.0,0.924271,16.0,0.812500,0.000000,17.0,3.0
8075,2706,28.0,0.884787,16.0,0.800705,0.000739,18.0,3.0
13335,7575,40.0,0.905543,16.0,0.812500,0.000000,14.0,3.0
15169,223703,16.0,0.800705,16.0,0.812500,0.000000,20.0,3.0
16509,1113,40.0,0.921570,16.0,0.820160,0.000000,17.0,3.0


In [28]:
cid_features = cid_features.merge(
    asn_profile,
    on="DST_ASN",
    how="left"
)

In [29]:
cid_features["known_asn"] = (
    cid_features["flows"].notna().astype(int)
)

In [30]:
global_scid_length = cid_features["scid_length"].median()

global_scid_entropy = cid_features["scid_entropy"].median()

global_oscid_length = cid_features["oscid_length"].median()

global_oscid_entropy = cid_features["oscid_entropy"].median()

global_retry_rate = cid_features["retry_present"].mean()

global_sni_length = cid_features["sni_length"].median()

global_sni_labels = cid_features["sni_labels"].median()

In [31]:
fill_values = {
    "median_scid_length": global_scid_length,
    "median_scid_entropy": global_scid_entropy,
    "median_oscid_length": global_oscid_length,
    "median_oscid_entropy": global_oscid_entropy,
    "retry_rate": global_retry_rate,
    "median_sni_length": global_sni_length,
    "median_sni_labels": global_sni_labels
}

cid_features = cid_features.fillna(fill_values)

In [32]:
asn_profile.describe().T

,count,mean,std,min,25%,50%,75%,max
flows,12.0,23914.500000,63420.931934,378.000000,1003.750000,2205.500000,10224.500000,223703.000000
median_scid_length,12.0,28.500000,11.571910,16.000000,16.000000,31.000000,40.000000,40.000000
median_scid_entropy,12.0,0.863242,0.064307,0.769455,0.800705,0.895165,0.922245,0.926550
median_oscid_length,12.0,16.000000,0.000000,16.000000,16.000000,16.000000,16.000000,16.000000
median_oscid_entropy,12.0,0.808862,0.007712,0.800705,0.800705,0.812500,0.812500,0.820160
retry_rate,12.0,0.000066,0.000213,0.000000,0.000000,0.000000,0.000000,0.000739
median_sni_length,12.0,20.333333,5.757735,14.000000,17.000000,18.000000,21.250000,33.000000
median_sni_labels,12.0,3.083333,0.288675,3.000000,3.000000,3.000000,3.000000,4.000000


In [33]:
asn_profile.sort_values("flows", ascending=False).head(20)

,flows,median_scid_length,median_scid_entropy,median_oscid_length,median_oscid_entropy,retry_rate,median_sni_length,median_sni_labels
DST_ASN,,,,,,,,
15169,223703,16.0,0.800705,16.0,0.812500,0.000000,20.0,3.0
32934,25499,16.0,0.769455,16.0,0.800705,0.000000,18.0,3.0
396982,18173,16.0,0.800705,16.0,0.800705,0.000055,29.0,3.0
13335,7575,40.0,0.905543,16.0,0.812500,0.000000,14.0,3.0
54113,3465,34.0,0.911428,16.0,0.812500,0.000000,20.0,4.0
8075,2706,28.0,0.884787,16.0,0.800705,0.000739,18.0,3.0
20940,1705,16.0,0.788910,16.0,0.812500,0.000000,25.0,3.0
36183,1374,40.0,0.924271,16.0,0.800705,0.000000,15.0,3.0
16509,1113,40.0,0.921570,16.0,0.820160,0.000000,17.0,3.0


In [34]:
cid_features["scid_length_deviation"] = (
    cid_features["scid_length"] -
    cid_features["median_scid_length"]
).abs()

cid_features["oscid_length_deviation"] = (
    cid_features["oscid_length"] -
    cid_features["median_oscid_length"]
).abs()

In [35]:
cid_features["sni_length_deviation"] = (
    cid_features["sni_length"] -
    cid_features["median_sni_length"]
).abs()

In [36]:
cid_features["scid_entropy_deviation"] = (
    cid_features["scid_entropy"] -
    cid_features["median_scid_entropy"]
).abs()

cid_features["oscid_entropy_deviation"] = (
    cid_features["oscid_entropy"] -
    cid_features["median_oscid_entropy"]
).abs()

In [37]:
cid_final = cid_features[
    [
        # Raw CID Features
        "occid_length",
        "oscid_length",
        "scid_length",
        "retry_present",
        "sni_length",
        "sni_labels",

        # Statistical Features
        "oscid_entropy",
        "scid_entropy",

        # Context Features
        "scid_length_deviation",
        "oscid_length_deviation",
        "scid_entropy_deviation",
        "oscid_entropy_deviation",
        "sni_length_deviation",
        "known_asn"
    ]
].copy()


print(cid_final.shape)

(287806, 14)


In [38]:
cid_final.head()

,occid_length,oscid_length,scid_length,retry_present,sni_length,sni_labels,oscid_entropy,scid_entropy,scid_length_deviation,oscid_length_deviation,scid_entropy_deviation,oscid_entropy_deviation,sni_length_deviation,known_asn
0,32,32,16,0,15,3,0.902115,0.843750,0.0,16.0,0.043045,0.089615,5.0,1
1,0,16,16,0,16,3,0.906250,0.800705,0.0,0.0,0.031250,0.105545,2.0,1
2,0,16,40,0,11,2,0.714615,0.924271,0.0,0.0,0.018728,0.097885,3.0,1
3,0,16,16,0,15,3,0.713054,0.831955,0.0,0.0,0.062500,0.087651,3.0,1
4,0,16,16,0,15,3,0.812500,0.788910,0.0,0.0,0.019455,0.011795,3.0,1


In [39]:
hist_features = df[
    [
        "PHIST_SRC_SIZES",
        "PHIST_DST_SIZES",
        "PHIST_SRC_IPT",
        "PHIST_DST_IPT"
    ]
].copy()

In [40]:
def histogram_entropy(hist):

    # Convert to NumPy array
    hist = np.array(hist, dtype=float)

    # Total observations
    total = hist.sum()

    # Handle empty histograms
    if total == 0:
        return 0

    # Convert counts to probabilities
    probabilities = hist / total

    # Calculate Shannon entropy
    entropy = 0

    for p in probabilities:
        if p > 0:
            entropy -= p * math.log2(p)

    # Normalize entropy
    max_entropy = math.log2(len(hist))

    return entropy / max_entropy

In [41]:
# Source packet size histogram entropy
hist_features["src_size_entropy"] = (
    hist_features["PHIST_SRC_SIZES"]
    .apply(histogram_entropy)
)

# Destination packet size histogram entropy
hist_features["dst_size_entropy"] = (
    hist_features["PHIST_DST_SIZES"]
    .apply(histogram_entropy)
)

# Source IPT histogram entropy
hist_features["src_ipt_entropy"] = (
    hist_features["PHIST_SRC_IPT"]
    .apply(histogram_entropy)
)

# Destination IPT histogram entropy
hist_features["dst_ipt_entropy"] = (
    hist_features["PHIST_DST_IPT"]
    .apply(histogram_entropy)
)

In [42]:
hist_features[
    [
        "src_size_entropy",
        "dst_size_entropy",
        "src_ipt_entropy",
        "dst_ipt_entropy"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
src_size_entropy,287806.0,0.588657,0.166159,0.0,0.518886,0.623846,0.705831,0.926880
dst_size_entropy,287806.0,0.596955,0.195563,0.0,0.552581,0.653655,0.718546,0.935785
src_ipt_entropy,287806.0,0.344961,0.180320,0.0,0.228013,0.330421,0.468546,0.986643
dst_ipt_entropy,287806.0,0.335694,0.202498,0.0,0.188837,0.328409,0.480385,1.000000


In [43]:
hist_final = hist_features[
    [
        "src_size_entropy",
        "dst_size_entropy",
        "src_ipt_entropy",
        "dst_ipt_entropy"
    ]
].copy()

print(hist_final.shape)

hist_final.head()

(287806, 4)


,src_size_entropy,dst_size_entropy,src_ipt_entropy,dst_ipt_entropy
0,0.525992,0.537034,0.103181,0.480563
1,0.656604,0.697424,0.267691,0.584478
2,0.586988,0.552581,0.167753,0.181188
3,0.270426,0.686271,0.528321,0.181188
4,0.431821,0.803867,0.408131,0.486383


In [44]:
flow_df = df.copy()

In [45]:
flow_df["duration_safe"] = flow_df["DURATION"].clip(lower=1e-6)

In [46]:
flow_df["total_bytes"] = (
    flow_df["BYTES"] +
    flow_df["BYTES_REV"]
)

In [47]:
flow_df["total_packets"] = (
    flow_df["PACKETS"] +
    flow_df["PACKETS_REV"]
)

In [48]:
flow_df["byte_rate"] = (
    flow_df["total_bytes"] /
    flow_df["duration_safe"]
)

In [49]:
flow_df["packet_rate"] = (
    flow_df["total_packets"] /
    flow_df["duration_safe"]
)

In [50]:
flow_df["avg_packet_size"] = (
    flow_df["total_bytes"] /
    flow_df["total_packets"].clip(lower=1)
)

In [51]:
flow_df["byte_ratio"] = (
    flow_df["BYTES"] /
    flow_df["BYTES_REV"].clip(lower=1)
)

In [52]:
flow_df["packet_ratio"] = (
    flow_df["PACKETS"] /
    flow_df["PACKETS_REV"].clip(lower=1)
)

In [53]:
flow_features = flow_df[
    [
        "DURATION",
        "FLOW_END_REASON",
        "total_bytes",
        "total_packets",
        "byte_rate",
        "packet_rate",
        "avg_packet_size",
        "byte_ratio",
        "packet_ratio"
    ]
]

In [54]:
flow_features.replace([np.inf, -np.inf], np.nan, inplace=True)

flow_features.isnull().sum()

DURATION           0
FLOW_END_REASON    0
total_bytes        0
total_packets      0
byte_rate          0
packet_rate        0
avg_packet_size    0
byte_ratio         0
packet_ratio       0
dtype: int64

In [55]:
flow_features = flow_df[
    [
        "DURATION",
        "FLOW_END_REASON",
        "total_bytes",
        "total_packets",
        "byte_rate",
        "packet_rate",
        "avg_packet_size",
        "byte_ratio",
        "packet_ratio"
    ]
].copy()

In [56]:
flow_features["log_duration"] = np.log1p(flow_features["DURATION"])

flow_features["log_total_bytes"] = np.log1p(flow_features["total_bytes"])

flow_features["log_total_packets"] = np.log1p(flow_features["total_packets"])

flow_features["log_byte_rate"] = np.log1p(flow_features["byte_rate"])

flow_features["log_packet_rate"] = np.log1p(flow_features["packet_rate"])

flow_features["log_byte_ratio"] = np.log1p(flow_features["byte_ratio"])

flow_features["log_packet_ratio"] = np.log1p(flow_features["packet_ratio"])

In [57]:
log_columns = [

"log_duration",

"FLOW_END_REASON",

"log_total_bytes",

"log_total_packets",

"log_byte_rate",

"log_packet_rate",

"avg_packet_size",

"log_byte_ratio",

"log_packet_ratio"

]

flow_features[log_columns].corr()

,log_duration,FLOW_END_REASON,log_total_bytes,log_total_packets,log_byte_rate,log_packet_rate,avg_packet_size,log_byte_ratio,log_packet_ratio
log_duration,1.000000,0.148644,0.517693,0.586430,-0.775497,-0.810360,0.068585,-0.016034,-0.185238
FLOW_END_REASON,0.148644,1.000000,0.088605,0.101976,-0.076132,-0.072823,0.010917,-0.004086,-0.023622
log_total_bytes,0.517693,0.088605,1.000000,0.970313,0.016806,-0.089938,0.583921,-0.191939,-0.438469
log_total_packets,0.586430,0.101976,0.970313,1.000000,-0.096102,-0.166919,0.385323,-0.156438,-0.372393
log_byte_rate,-0.775497,-0.076132,0.016806,-0.096102,1.000000,0.980414,0.379140,-0.091358,-0.101094
log_packet_rate,-0.810360,-0.072823,-0.089938,-0.166919,0.980414,1.000000,0.216261,-0.058967,-0.025939
avg_packet_size,0.068585,0.010917,0.583921,0.385323,0.379140,0.216261,1.000000,-0.214591,-0.490815
log_byte_ratio,-0.016034,-0.004086,-0.191939,-0.156438,-0.091358,-0.058967,-0.214591,1.000000,0.681993
log_packet_ratio,-0.185238,-0.023622,-0.438469,-0.372393,-0.101094,-0.025939,-0.490815,0.681993,1.000000


In [58]:
selected_columns = [

    "log_duration",
    "FLOW_END_REASON",
    "log_total_bytes",
    "log_byte_rate",
    "avg_packet_size",
    "log_byte_ratio",
    "log_packet_ratio"

]

flow_features_selected = flow_features[selected_columns].copy()

flow_features_selected.head()

,log_duration,FLOW_END_REASON,log_total_bytes,log_byte_rate,avg_packet_size,log_byte_ratio,log_packet_ratio
0,1.803304,1,10.615285,8.992109,849.000000,0.458604,0.503905
1,1.647295,1,9.565214,8.132042,419.382353,0.585601,0.818310
2,0.158655,1,8.756210,10.516724,334.157895,1.036035,0.747214
3,0.076998,1,8.722580,11.247659,472.230769,0.967936,0.367725
4,0.249948,1,9.510963,10.769836,587.217391,0.851155,0.570545


In [59]:
quic_features = df[
    [
        "QUIC_VERSION",
        "QUIC_CLIENT_VERSION",
        "QUIC_TOKEN_LENGTH",
        "QUIC_ZERO_RTT",
        "QUIC_MULTIPLEXED"
    ]
].copy()

quic_features.head()

,QUIC_VERSION,QUIC_CLIENT_VERSION,QUIC_TOKEN_LENGTH,QUIC_ZERO_RTT,QUIC_MULTIPLEXED
0,1,1,0,5,2
1,1,1,0,0,0
2,1,1,0,1,0
3,1,1,0,0,0
4,4207849474,4207849474,0,0,0


In [60]:
quic_features["token_present"] = (
    quic_features["QUIC_TOKEN_LENGTH"] > 0
).astype(int)

In [61]:
quic_features["log_token_length"] = np.log1p(
    quic_features["QUIC_TOKEN_LENGTH"]
)

In [62]:
quic_features["zero_rtt_present"] = (
    quic_features["QUIC_ZERO_RTT"] > 0
).astype(int)

In [63]:
quic_features["multiplexed"] = (
    quic_features["QUIC_MULTIPLEXED"] > 0
).astype(int)

In [64]:
version_map = {
    version: idx
    for idx, version in enumerate(
        sorted(quic_features["QUIC_VERSION"].unique())
    )
}

quic_features["quic_version_id"] = (
    quic_features["QUIC_VERSION"].map(version_map)
)

In [65]:
version_map

{np.int64(0): 0,
 np.int64(1): 1,
 np.int64(3467641594): 2,
 np.int64(4207849474): 3,
 np.int64(4207849486): 4,
 np.int64(4207849491): 5,
 np.int64(4278190109): 6}

In [66]:
quic_features["log_zero_rtt"] = np.log1p(
    quic_features["QUIC_ZERO_RTT"]
)

In [67]:
quic_features["log_multiplexed"] = np.log1p(
    quic_features["QUIC_MULTIPLEXED"]
)

In [68]:
quic_selected = quic_features[
    [
        "quic_version_id",
        "log_token_length",
        "log_zero_rtt",
        "zero_rtt_present",
        "log_multiplexed",
        "multiplexed"
    ]
]

quic_selected.corr()

,quic_version_id,log_token_length,log_zero_rtt,zero_rtt_present,log_multiplexed,multiplexed
quic_version_id,1.000000,0.130246,-0.105831,-0.120746,-0.042778,-0.044244
log_token_length,0.130246,1.000000,0.651149,0.732223,0.002492,0.000704
log_zero_rtt,-0.105831,0.651149,1.000000,0.888653,-0.050350,-0.069363
zero_rtt_present,-0.120746,0.732223,0.888653,1.000000,-0.082610,-0.095075
log_multiplexed,-0.042778,0.002492,-0.050350,-0.082610,1.000000,0.960755
multiplexed,-0.044244,0.000704,-0.069363,-0.095075,0.960755,1.000000


In [69]:
selected_quic = quic_features[
    [
        "quic_version_id",
        "QUIC_TOKEN_LENGTH",
        "log_zero_rtt",
        "zero_rtt_present",
        "log_multiplexed",
        "multiplexed"
    ]
].copy()

selected_quic.head()

,quic_version_id,QUIC_TOKEN_LENGTH,log_zero_rtt,zero_rtt_present,log_multiplexed,multiplexed
0,1,0,1.791759,1,1.098612,1
1,1,0,0.000000,0,0.000000,0
2,1,0,0.693147,1,0.000000,0
3,1,0,0.000000,0,0.000000,0
4,3,0,0.000000,0,0.000000,0


In [70]:
with_CID_df = pd.concat([ppi_final,cid_final,hist_final,flow_features_selected,selected_quic],axis=1)

print(with_CID_df.shape)

with_CID_df.to_parquet(
    "with_CID.parquet",
    index=False
)

(287806, 40)


In [71]:
check_df = pd.read_parquet("with_CID.parquet")

print(check_df.shape)
check_df.head()

(287806, 40)


,PPI_LEN,PPI_ROUNDTRIPS,log_ppi_duration,log_mean_ipt,log_std_ipt,mean_packet_size,std_packet_size,direction_change_ratio,forward_packet_ratio,occid_length,...,log_byte_rate,avg_packet_size,log_byte_ratio,log_packet_ratio,quic_version_id,QUIC_TOKEN_LENGTH,log_zero_rtt,zero_rtt_present,log_multiplexed,multiplexed
0,30,7,0.031499,0.725937,1.249640,966.433333,496.378262,0.482759,0.466667,32,...,8.992109,849.000000,0.458604,0.503905,1,0,1.791759,1,1.098612,1
1,30,5,1.592902,4.879767,6.445720,431.633333,497.956055,0.310345,0.566667,0,...,8.132042,419.382353,0.585601,0.818310,1,0,0.000000,0,0.000000,0
2,19,6,0.158712,2.307834,3.386832,306.157895,459.089518,0.666667,0.526316,0,...,10.516724,334.157895,1.036035,0.747214,1,0,0.693147,1,0.000000,0
3,13,3,0.076961,1.967650,2.883625,444.230769,517.949085,0.416667,0.307692,0,...,11.247659,472.230769,0.967936,0.367725,1,0,0.000000,0,0.000000,0
4,23,5,0.249980,2.591354,3.277578,559.217391,538.014624,0.409091,0.434783,0,...,10.769836,587.217391,0.851155,0.570545,3,0,0.000000,0,0.000000,0


In [72]:
without_CID_df = pd.concat([ppi_final,hist_final,flow_features_selected,selected_quic],axis=1)

print(without_CID_df.shape)

without_CID_df.to_parquet(
    "without_CID.parquet",
    index=False
)

(287806, 26)


In [73]:
check_df2 = pd.read_parquet("without_CID.parquet")

print(check_df2.shape)
check_df2.head()

(287806, 26)


,PPI_LEN,PPI_ROUNDTRIPS,log_ppi_duration,log_mean_ipt,log_std_ipt,mean_packet_size,std_packet_size,direction_change_ratio,forward_packet_ratio,src_size_entropy,...,log_byte_rate,avg_packet_size,log_byte_ratio,log_packet_ratio,quic_version_id,QUIC_TOKEN_LENGTH,log_zero_rtt,zero_rtt_present,log_multiplexed,multiplexed
0,30,7,0.031499,0.725937,1.249640,966.433333,496.378262,0.482759,0.466667,0.525992,...,8.992109,849.000000,0.458604,0.503905,1,0,1.791759,1,1.098612,1
1,30,5,1.592902,4.879767,6.445720,431.633333,497.956055,0.310345,0.566667,0.656604,...,8.132042,419.382353,0.585601,0.818310,1,0,0.000000,0,0.000000,0
2,19,6,0.158712,2.307834,3.386832,306.157895,459.089518,0.666667,0.526316,0.586988,...,10.516724,334.157895,1.036035,0.747214,1,0,0.693147,1,0.000000,0
3,13,3,0.076961,1.967650,2.883625,444.230769,517.949085,0.416667,0.307692,0.270426,...,11.247659,472.230769,0.967936,0.367725,1,0,0.000000,0,0.000000,0
4,23,5,0.249980,2.591354,3.277578,559.217391,538.014624,0.409091,0.434783,0.431821,...,10.769836,587.217391,0.851155,0.570545,3,0,0.000000,0,0.000000,0


In [74]:
print(with_CID_df.columns.duplicated().sum())
print(without_CID_df.columns.duplicated().sum())

0
0
